In [20]:
import numpy as np
from collections import Counter

In [16]:
examples = [
                    {'Outlook': 'Sunny', 'Temperature': 'Hot', 'Humidity': 'High', 'Wind': 'Weak', 'PlayTennis': 'No'},
                    {'Outlook': 'Sunny', 'Temperature': 'Hot', 'Humidity': 'High', 'Wind': 'Strong', 'PlayTennis': 'No'},
                    {'Outlook': 'Overcast', 'Temperature': 'Hot', 'Humidity': 'High', 'Wind': 'Weak', 'PlayTennis': 'Yes'},
                    {'Outlook': 'Rain', 'Temperature': 'Mild', 'Humidity': 'High', 'Wind': 'Weak', 'PlayTennis': 'Yes'}
                ]
attributes = ['Outlook', 'Temperature', 'Humidity', 'Wind']

In [18]:
[d['PlayTennis'] for d in examples]

['No', 'No', 'Yes', 'Yes']

In [19]:
class TreeNode:
    def __init__(self, left = None, right = None, is_leaf = False, feature = None, split_value = None, prediction = None):
        self.left = left
        self.right = right
        self.is_leaf = is_leaf
        self.feature = feature
        self.split_value = split_value
        self.prediction = prediction

In [ ]:
class DecisionTree:
    def __init__(self, examples, attributes, target):
        self.examples = examples
        self.attributes = attributes
        self.target = target
        self.root = TreeNode()
        

    def getEntropy(self, data):
        target_cats = list(set([d[self.target] for d in data]))
        if len(target_cats) == 1:
            return 0
        else:
            res = 0
            n = len(data)
            for cat in target_cats:
                p = len([1 for d in data if d[self.target] == cat]) / n
                res -= p * np.log2(p)
        return res

    def entropyReduction(self, data, feature, split_value):
        '''
        Compute the information gain (entropy reduction) from splitting the dataset by the given feature and split_value.
        '''
        left_data = [d for d in data if d[feature] == split_value]
        right_data = [d for d in data if d[feature] != split_value]
        res = self.getEntropy(data) - self.getEntropy(left_data) * len(left_data)/len(data) - self.getEntropy(right_data) * len(right_data)/len(data)
        return res

    def getFeature(self, data, attributes):
        '''
        Return the best feature and split value for splitting
        '''
        best_result = (None, None, float('-inf'))
        for feature in attributes:
            values = list(set([d[feature] for d in data]))
            for val in values:
                res = self.entropyReduction(data, feature, val)
                if res > best_result[2]:
                    best_result = (feature, val, res)
        return best_result[0], best_result[1]

    def buildTree(self, data, attributes, node):
        '''
        Build desicion tree recursively.
        '''
        if len(set([d[self.target] for d in data])) == 1:
            # leaf node
            return TreeNode(prediction = data[0][self.target], is_leaf = True)

        # split data into left_data and right_data
        feature, split_value = self.getFeature(data, attributes)
        
        # update node
        node.feature = feature
        node.split_value = split_value
        left_data = [d for d in data if d[feature] == split_value]
        right_data = [d for d in data if d[feature] != split_value]
        if left_data == [] or right_data == []:
            node.is_left = True
            node.prediction = Counter([d[self.target] for d in data]).most_common(1)[0][0]
            return node

        node.left = self.buildTree(left_data, attributes)
        node.right = self.buildTree(right_data, attributes)

        return node